# Lesson 5 — Code Quality, Functional Programming, Testing
### Intro to Computer Science for Non-CS Students

**The bridge from Lessons 1–4.** Four threads run into this lesson:

1. **Lesson 1:** a structure is a bundle of **properties**, and a type is a
   claim about which operations are legitimate.
2. **Lesson 2:** pandas gave us verbs — load, clean, `groupby`, `merge` — and the
   habit of *interviewing the data before trusting it*.
3. **Lesson 3:** the terminal, virtual environments, and git — including the
   `add → commit` rhythm and `if __name__ == "__main__"`.
4. **Lesson 4:** algorithms and Big-O — scans, the *algorithm vs implementation*
   distinction, and the edge cases that break code: empty input, one element, ties.

So far the goal has been code that **works once**. This lesson is about code you
can **trust twice** — reuse next month, hand to a colleague, and check
automatically.

```
Code that runs once            "it printed the right number today"
        ↓
Functions as contracts   (Unit 1)   name the inputs, the promise, the return
        ↓
Functional tools         (Unit 2)   comprehensions, sorted, chains, vectorization
        ↓
Refactoring & modules    (Unit 3)   kill the smells, split into files you import
        ↓
Errors & defensive code  (Unit 4)   read tracebacks, fail loudly at the door
        ↓
Testing with pytest      (Unit 5)   prove it, automatically — twice
```

**Prerequisites and setup**

- Lessons 1–4: types and properties, pandas, the terminal and git, Big-O.
- Uses `pandas` and `pytest`, the same course environment as earlier lessons.
- Runs top to bottom, offline; later cells depend on earlier ones.
- Everything it creates at run time goes under `../data/` (gitignored) — never in
  the repo tree.

**Suggested sessions:** ① Units 1–2 · ② Units 3–4 · ③ Unit 5

**Practice** lives in `../lesson5_exercise/` — a working-but-awful script and a
pytest checker you make pass, committing each green step.

In [ ]:
import sys
import time
import traceback
from pathlib import Path

import pandas as pd

print("pandas", pd.__version__)

# This lesson writes a few small files (demo modules, a broken CSV). They ALL
# live under ../data/lesson5_demo/, which is gitignored — the Lesson 2 rule:
# never write run-time artifacts into the repo tree. pandas writes files, not
# folders, so we make the folder once, up front.
DEMO = Path("../data/lesson5_demo")
DEMO.mkdir(parents=True, exist_ok=True)

DATA = Path("../course_data/online_retail_sample.csv")
print("data file exists:", DATA.exists())
print("demo folder ready:", DEMO)

## Unit 1 — Functions as contracts  ·  ~45 min

The first tool for trustworthy code is the humble function, seen properly: not a
way to avoid retyping, but a **contract**. A contract has three visible parts:

```
Signature    — the name and the inputs: what you must supply
Docstring    — the promise: what it does and what it returns
Return value — the deliverable: the answer handed back to the caller
```

Read a function's first two lines and you should know how to *use* it without
reading its body. That is the whole idea.

In [ ]:
def revenue(quantity: int, unit_price: float) -> float:
    """Revenue for one order line: quantity times unit price."""
    return quantity * unit_price

# The caller reads the signature and the docstring, not the body.
print("revenue(3, 2.5) =", revenue(3, 2.5))
print("the promise    :", revenue.__doc__)

### Return, don't print

A function that *prints* its answer has thrown it away — the caller gets `None`
and cannot reuse it. A function that *returns* its answer hands it back, so it
can be stored, combined, and (Unit 5) tested. Lesson 4 made the same point about
`max()`: the value is the deliverable, not the print.

In [ ]:
def print_revenue(quantity, unit_price):
    print(quantity * unit_price)          # shows the number, returns nothing

def return_revenue(quantity, unit_price):
    return quantity * unit_price          # hands the number back

shown = print_revenue(3, 2.5)             # prints 7.5 as a side effect...
handed = return_revenue(3, 2.5)           # ...prints nothing

print("print_revenue gave back:", shown)   # None — nothing to reuse
print("return_revenue gave back:", handed)
print("only the returned value composes:", handed * 2)

### Pure vs side-effecting

A **pure** function depends only on its inputs and changes nothing outside
itself: same inputs → same output, every time. A **side-effecting** function
reaches outside — it prints, writes a file, or *modifies its argument*.

Side effects are not evil (writing your cleaned file is a side effect you want).
But a function that *silently mutates the data you passed in* is a trap: the
caller's DataFrame changes under their feet.

In [ ]:
def drop_cheap_MUTATING(orders):
    """Looks innocent — but edits the caller's frame in place (a side effect)."""
    orders.drop(orders[orders["price"] < 1.0].index, inplace=True)
    return orders

def drop_cheap_PURE(orders):
    """Returns a NEW frame; the caller's data is left untouched."""
    return orders[orders["price"] >= 1.0].reset_index(drop=True)

toy = pd.DataFrame({"item": ["a", "b", "c"], "price": [0.5, 2.0, 3.0]})

drop_cheap_PURE(toy)
print("after PURE call,     original still has", len(toy), "rows")   # 3 — safe

drop_cheap_MUTATING(toy)
print("after MUTATING call, original now has  ", len(toy), "rows")   # 2 — surprise

🧭 This is not a toy worry. The exercise at the end of the lesson has a function
`clean_prices` whose contract reads *"Return a NEW frame ... Must not modify the
frame passed in."* — a pure function, on purpose, because a mutating one would
corrupt every step downstream.

### Type hints are property declarations

`def revenue(quantity: int, unit_price: float) -> float:` says out loud: *I take
a whole number and a decimal; I give back a decimal.* That is Lesson 1's central
idea — the type is a claim about which operations are legitimate — written into
the signature. Python does **not** enforce hints at run time; they are
documentation that humans read and tools (editors, type checkers) verify. A hint
like `-> pd.DataFrame` promises the caller a table, not a number.

### Refactor a long cell into three named functions

Here is a Lesson-2-style cleaning cell: load, clean, and summarise all tangled
together, printing as it goes. It works — and it is exactly the shape that grows
unmanageable.

In [ ]:
# BEFORE — one long procedural paragraph.
raw = pd.read_csv("../course_data/online_retail_sample.csv")
raw["Description"] = raw["Description"].str.strip()
raw["revenue"] = raw["Quantity"] * raw["UnitPrice"]
raw = raw[raw["UnitPrice"] >= 1.0].reset_index(drop=True)
print("rows kept:", len(raw))
print("total revenue:", round(raw["revenue"].sum(), 2))

In [ ]:
# AFTER — three named contracts. Each does one job and RETURNS its result.
def load_orders(path: str) -> pd.DataFrame:
    """Read the retail CSV, tidy Description, add a revenue column. Keep all rows."""
    orders = pd.read_csv(path)
    orders["Description"] = orders["Description"].str.strip()
    orders["revenue"] = orders["Quantity"] * orders["UnitPrice"]
    return orders

def keep_real_prices(orders: pd.DataFrame) -> pd.DataFrame:
    """Return a NEW frame with only rows priced at 1.00 or more (a pure step)."""
    return orders[orders["UnitPrice"] >= 1.0].reset_index(drop=True)

def total_revenue(orders: pd.DataFrame) -> float:
    """Grand total of the revenue column — a Lesson-4 scan — rounded to 2 dp."""
    return round(orders["revenue"].sum(), 2)

orders = keep_real_prices(load_orders("../course_data/online_retail_sample.csv"))
print("rows kept:", len(orders))
print("total revenue:", total_revenue(orders))

🧭 Same numbers — but now each step has a name you can read, a docstring you can
trust, and a return value you can test. `keep_real_prices` is *pure*: call it a
hundred times and the input never changes. Hold onto these three; the final
exercise asks you to carve up a much uglier script exactly this way.

> **Unit 1 takeaway:** a function is a contract — signature, docstring, return
> value. Prefer returning over printing, prefer pure over mutating, and let type
> hints state the properties you expect.

## Unit 2 — Functional tools  ·  ~50 min

Lesson 4 showed that most analytics is scanning, filtering, and combining. Python
has a compact, readable vocabulary for exactly those moves — the "functional"
toolkit. You have used pieces of it already; today we name them deliberately and
reach the punchline: **a pandas method chain is a functional pipeline.**

### Comprehensions, revisited on purpose

Lesson 4 met the list comprehension as a one-line filter. It is really *map +
filter in one breath*: for each item, optionally keep it, optionally transform it.

In [ ]:
prices = [0.5, 2.0, 3.5, 0.8, 5.0]

# list: keep prices >= 1, express in pence (filter + transform together)
pence = [round(p * 100) for p in prices if p >= 1.0]
print("list  :", pence)

# dict: build a lookup (Lesson 4's O(1) friend)
price_label = {p: ("cheap" if p < 1.0 else "ok") for p in prices}
print("dict  :", price_label)

# set: unique first letters (uniqueness is a Lesson 1 property)
words = ["metal", "mug", "doormat", "mug"]
print("set   :", {w[0] for w in words})

### `sorted` with `key=` — and the tie-break trick

Lesson 4 built leaderboards with `sorted(..., key=lambda r: r["revenue"])`. The
`key` says *what to compare records by*. A **tuple key** sorts by several things
at once — the exact tool for "highest total, ties broken by the smaller id."

In [ ]:
customers = [
    {"id": 111, "total": 50.0},
    {"id": 999, "total": 50.0},
    {"id": 555, "total": 50.0},
    {"id": 222, "total": 90.0},
]

# highest total first; when totals tie, smaller id first.
ranked = sorted(customers, key=lambda c: (-c["total"], c["id"]))
for c in ranked:
    print(c["id"], c["total"])

🧭 Read the key `(-c["total"], c["id"])` as *"sort by total **descending**, then
by id **ascending**"* — negating one field flips its direction. This is precisely
how the exercise's `top_customers` breaks ties. A `lambda` is just a one-line
unnamed function (Lesson 4); when the logic outgrows a line, use a named `def`.

### `map` / `filter` vs comprehensions

`map(f, xs)` applies `f` to every item; `filter(pred, xs)` keeps the ones where
`pred` is true. Same two moves as a comprehension — and in Python the
comprehension usually reads better.

In [ ]:
nums = [1, 2, 3, 4, 5, 6]

with_map_filter = list(map(lambda n: n * 2, filter(lambda n: n % 2 == 0, nums)))
with_comprehension = [n * 2 for n in nums if n % 2 == 0]

print("map + filter :", with_map_filter)
print("comprehension:", with_comprehension)
print("same result  :", with_map_filter == with_comprehension)

### `zip` and `enumerate` — walking in step and counting

`zip` walks two sequences together (Lesson 4's "build a lookup" often starts
here); `enumerate` hands you a running position — the clean way to print a ranked
list, no manual counter.

In [ ]:
ids = [111, 222, 555]
totals = [50.0, 90.0, 50.0]

lookup = {cid: total for cid, total in zip(ids, totals)}   # zip -> dict
print("zipped lookup:", lookup)

for rank, c in enumerate(ranked, start=1):     # 1-based ranks, no counter variable
    print(f"#{rank}: customer {c['id']} spent {c['total']}")

### The bridge: a pandas chain *is* a functional pipeline

Each of Lesson 2's verbs takes a DataFrame and returns a **new** one. Chain them
and you have a pipeline — data flowing through pure steps — the same idea as
function composition, but readable top to bottom.

In [ ]:
# step-by-step, a temporary variable per step
step1 = orders[orders["Country"] == "United Kingdom"]
step2 = step1.groupby("StockCode")["revenue"].sum()
step3 = step2.sort_values(ascending=False)
top_uk_stepwise = step3.head(3)

# the same pipeline as one chain — each method returns a new frame/series
top_uk_chain = (
    orders
    .query("Country == 'United Kingdom'")
    .groupby("StockCode")["revenue"].sum()
    .sort_values(ascending=False)
    .head(3)
)
print(top_uk_chain)
print("chain matches step-by-step:", top_uk_stepwise.equals(top_uk_chain))

🧭 Every link is a pure transformation: it returns a new structure and the
previous one survives (Lesson 1). Parentheses let you break a chain across lines.
Don't over-chain, though — if a step needs a comment or you want to inspect it, a
named variable is clearer than a ten-method run-on.

### `.apply` vs vectorization — and when each is defensible

Lesson 4's punchline: convenience hides the procedure, not the cost. `.apply(...,
axis=1)` runs your Python function once **per row** — an explicit O(n) Python
loop. Vectorized column arithmetic does the same work in compiled code, on the
whole column at once (Lesson 1's *homogeneity* property is what makes that legal).

In [ ]:
big = pd.concat([orders] * 300, ignore_index=True)   # ~15k rows, to make timing visible
print("rows:", len(big))

start = time.perf_counter()
apply_rev = big.apply(lambda row: row["Quantity"] * row["UnitPrice"], axis=1)
apply_s = time.perf_counter() - start

start = time.perf_counter()
vector_rev = big["Quantity"] * big["UnitPrice"]
vector_s = time.perf_counter() - start

print(f"  .apply row-wise : {apply_s:8.4f} s")
print(f"  vectorized      : {vector_s:8.5f} s")
print(f"  same answer     : {bool((apply_rev == vector_rev).all())}")
print(f"  speedup         : ~{apply_s / vector_s:,.0f}x")

🧭 Identical answers; the vectorized version is dramatically faster because it
never runs a Python-level loop. When is `.apply` defensible? When the per-row work
genuinely can't be vectorized — calling an external service per row, or logic that
weaves across several columns with no column-wise equivalent. Reach for
vectorization first; use `.apply` as an honest fallback, knowing its price.

> **Unit 2 takeaway:** comprehensions, `sorted(key=...)`, `zip`/`enumerate`, and
> `map`/`filter` are the readable functional toolkit; a pandas method chain is that
> same idea on tables; and vectorization beats `.apply` by swapping a Python loop
> for a compiled one.

---
## ⏱️ Session 2 warm-up  ·  5 questions from last time

Answer from memory first, then check.

1. A function is a contract with three visible parts — name them.
2. Why prefer returning a value over printing it?
3. What makes a function *pure*, and why does the exercise's `clean_prices` insist on it?
4. Read the sort key `(-total, id)` in plain English.
5. `df.apply(..., axis=1)` and `df["a"] * df["b"]` give the same answer — why prefer the second?

<details><summary><b>Answers</b></summary>

1. Signature (name + inputs), docstring (the promise), and return value (the deliverable).
2. A returned value can be stored, combined, and tested; a printed one is thrown
   away and the caller gets `None`.
3. It changes nothing outside itself and never modifies its argument — same input,
   same output. `clean_prices` must be pure so cleaning doesn't corrupt the frame
   every later step still reads.
4. Sort by total *descending*, and for ties, by id *ascending* (the negation flips
   total's direction).
5. `.apply(axis=1)` is a Python loop over rows; the vectorized product runs in
   compiled code on the whole column at once — same result, far cheaper.
</details>

## Unit 3 — Naming, structure, and refactoring  ·  ~45 min

Now point the tools at ugly code — specifically the kind *you* will write under
deadline. The exercise ships a script, `messy_analysis.py`: it works, it prints
correct numbers, and it commits every sin this unit names. Read it first.

In [ ]:
specimen = Path("../lesson5_exercise/messy_analysis.py").read_text()
print(specimen)

### Four code smells, caught red-handed

| Smell | In the specimen | Why it hurts |
| --- | --- | --- |
| **Copy-paste blocks** | three near-identical country loops (UK, EIRE, Netherlands) | change the rule once → you must fix it in three places → you miss one |
| **Magic numbers** | `>= 1.0`, unexplained | what is `1.0`? a price floor? why that value? the code won't say |
| **Deep nesting** | the top-3 selection loop, three `if`s deep | hard to read, easy to get the tie-break subtly wrong |
| **Print, not return** | every result is `print`ed, nothing returned | you can't reuse or test a number that was only printed (Unit 1) |

None of these is a *bug* — the script is correct. They are what makes correct code
impossible to trust, extend, or test. The cure is to **extract functions**.

### Extract one function — into a module you import

The three country loops all do the same thing: total `revenue` for rows matching a
country. That is a `groupby` (Lesson 2) — one function replacing the copy-paste.
We'll extract it (plus the load and clean steps) into its own **module file**, then
import it. This is your first multi-file project. `%%writefile` writes the cell's
body to disk instead of running it:

In [ ]:
%%writefile ../data/lesson5_demo/retail_utils.py
"""Retail helpers, extracted from messy_analysis.py. Same numbers, better shape."""
import pandas as pd

PRICE_FLOOR = 1.0   # the magic 1.0 from the script, named once

def load_orders(path):
    """Read the CSV, tidy Description, add revenue. Keep every row."""
    orders = pd.read_csv(path)
    orders["Description"] = orders["Description"].str.strip()
    orders["revenue"] = orders["Quantity"] * orders["UnitPrice"]
    return orders

def keep_real_prices(orders):
    """Return a NEW frame of rows priced at or above PRICE_FLOOR (pure)."""
    return orders[orders["UnitPrice"] >= PRICE_FLOOR].reset_index(drop=True)

def revenue_by_country(orders):
    """The three copy-pasted country loops, collapsed into one groupby."""
    return orders.groupby("Country")["revenue"].sum().round(2).to_dict()

In [ ]:
# A module is found on sys.path. Our demo module lives in ../data/lesson5_demo/,
# so we add that folder, then import from it like any library.
sys.path.insert(0, "../data/lesson5_demo")
from retail_utils import load_orders, keep_real_prices, revenue_by_country

demo_orders = keep_real_prices(load_orders("../course_data/online_retail_sample.csv"))
print(revenue_by_country(demo_orders))

🧭 Compare with the messy script's printed figures: **United Kingdom 1220.07, EIRE
148.88, Netherlands 16.45** — identical, but three loops became one function, now
importable and testable. The import line `from retail_utils import
revenue_by_country` is itself a contract: *there is a module `retail_utils`, and it
promises a `revenue_by_country` I may call.* (In a real project the module would
sit next to your notebook; here it lives under the gitignored `../data/` so the repo
stays clean.)

> **Unit 3 takeaway:** name your constants, collapse copy-paste into one function,
> keep nesting shallow, and split reusable code into a module you import.
> Refactoring changes the *shape* of code, never its answers.

## Unit 4 — Errors and defensive code  ·  ~35 min

Refactored code still meets bad input. This unit is about failing *well*: reading
the error Python hands you, catching the narrow thing you expected, and raising a
clear error the instant a contract is broken.

### Read tracebacks bottom-up

A traceback is a stack of "who called whom." The **bottom line** is the actual
error; the lines above are the path that led there. Read the bottom first
(Lesson 2 did this with `FileNotFoundError`), then walk up only if you need the
*where*.

In [ ]:
def parse_price(text):
    return float(text)                        # blows up on "not-available"

def load_prices(values):
    return [parse_price(v) for v in values]   # calls parse_price per item

try:
    load_prices(["1.95", "not-available", "2.10"])
except ValueError:
    print(traceback.format_exc())             # show the real traceback, safely

🧭 The bottom line is the fact: `ValueError: could not convert string to float:
'not-available'`. The frames above are the *where*: `load_prices` called
`parse_price`, which raised. Bottom = what went wrong; up = where it happened.

### `try` / `except` for the narrow case

Catch the *specific* exception you can handle — not everything. A bare `except:`
swallows real bugs (typos, wrong types) behind a message you wrote for a different
problem.

In [ ]:
def read_table(path):
    """Load a CSV, but turn a missing file into a clear, actionable message."""
    try:
        return pd.read_csv(path)
    except FileNotFoundError:
        raise FileNotFoundError(
            f"No file at {path!r} — check the path and your working directory "
            "(Lesson 2's first aid: print(Path.cwd()))."
        )

try:
    read_table("../course_data/does_not_exist.csv")
except FileNotFoundError as error:
    print("handled:", error)

We catch **only** `FileNotFoundError`. A different failure — a permissions error, a
malformed file — is *not* swallowed; it surfaces, which is exactly what you want.

### Raise when a contract is violated — validate early

Unit 1 said a function is a contract. The strongest way to keep one is to check the
promise **at the door** and raise immediately if it's broken, with a message that
names what was expected. This is Lesson 2's "interview the DataFrame before
trusting it," now enforced in code instead of eyeballed.

In [ ]:
REQUIRED = {"Quantity", "UnitPrice", "Country", "source_customer_id"}

def load_validated(path):
    """Load the retail CSV and FAIL LOUDLY if it isn't shaped as promised."""
    orders = pd.read_csv(path)
    missing = REQUIRED - set(orders.columns)
    if missing:
        raise ValueError(f"{path}: missing required columns {sorted(missing)}")
    if len(orders) == 0:
        raise ValueError(f"{path}: file has a header but zero rows")
    return orders

# happy path
good = load_validated("../course_data/online_retail_sample.csv")
print("loaded", len(good), "valid rows")

# broken contract: save a frame that is missing a required column, then reload it
broken_path = "../data/lesson5_demo/broken.csv"
good.drop(columns=["UnitPrice"]).to_csv(broken_path, index=False)
try:
    load_validated(broken_path)
except ValueError as error:
    print("caught early:", error)

🧭 The second call never reaches an analysis — it fails at load, naming the missing
column. A validated failure at the door beats a mysterious `KeyError` five cells
later. (This is exactly how the finance assignment's checker guards its inputs —
you'll see that file in Unit 5.)

> **Unit 4 takeaway:** read tracebacks bottom-up, catch the narrow exception you
> expected, and raise `ValueError` the moment a contract is broken — check the
> data's properties at load time and fail loudly.

---
## ⏱️ Session 3 warm-up  ·  5 questions from last time

1. Name three of the four code smells in `messy_analysis.py`.
2. What does `from retail_utils import revenue_by_country` promise?
3. Refactoring changed the code's *shape* — what must it never change?
4. In a traceback, which line is the actual error?
5. Why raise a `ValueError` at load time instead of letting the analysis crash later?

<details><summary><b>Answers</b></summary>

1. Any three of: copy-paste blocks (the country loops), magic numbers (`1.0`),
   deep nesting (the top-3 loop), and print-instead-of-return.
2. That a module named `retail_utils` exists and offers a `revenue_by_country`
   function to call — the import is a contract.
3. The answers — the numbers the code computes must stay identical.
4. The bottom line; the frames above only tell you where it happened.
5. A clear failure at the door names the real problem; a late crash surfaces far
   from the cause, as a cryptic `KeyError` or a wrong result.
</details>

## Unit 5 — Testing with pytest  ·  ~50 min

Here is the payoff of the whole lesson — and of Lesson 3's terminal work. Your
functions return values (Unit 1), so you can now *check those values
automatically*. A test is a tiny experiment: set up known inputs, run the
function, assert the answer. Do it once by hand, then let `pytest` run it for you
from the command line.

### `assert` — the atom of testing

`assert <condition>, "message"` does nothing when the condition is true and raises
`AssertionError` with your message when it's false. That is the entire mechanism
underneath every test framework.

In [ ]:
def discounted(price, fraction):
    """Price after a fractional discount (0.10 = 10% off), rounded to 2 dp."""
    return round(price * (1 - fraction), 2)

assert discounted(100.0, 0.10) == 90.0        # passes silently
assert discounted(50.0, 0.0) == 50.0          # no discount, no change

# a failing assert, shown safely so the notebook keeps running
try:
    assert discounted(100.0, 0.10) == 91.0, "expected 90.0, got a different value"
except AssertionError as error:
    print("AssertionError:", error)

### Test functions — arrange, act, assert

Wrap each check in a function named `test_...`. Inside, follow one rhythm:

```
Arrange — build the known inputs
Act     — call the function under test
Assert  — state the expected answer
```

`pytest` discovers every `test_...` function and runs it. We'll write a tiny module
and its tests to disk, then run pytest against them — the way real projects are
laid out.

In [ ]:
%%writefile ../data/lesson5_demo/discounts.py
"""A tiny module to test: fractional discounts and a safe average."""

def discounted(price, fraction):
    """Price after a fractional discount (0.10 = 10% off), rounded to 2 dp."""
    return round(price * (1 - fraction), 2)

def average(values):
    """Mean of a list of numbers. Empty list -> 0.0 (our chosen convention)."""
    if not values:                 # the empty-input edge case (Lesson 4)
        return 0.0
    return round(sum(values) / len(values), 2)

In [ ]:
%%writefile ../data/lesson5_demo/test_discounts.py
"""Tests for discounts.py — run with:  pytest test_discounts.py -v"""
from discounts import average, discounted

def test_discounted_ten_percent():
    price, fraction = 100.0, 0.10        # arrange
    result = discounted(price, fraction) # act
    assert result == 90.0                # assert

def test_discounted_zero_is_no_change():
    assert discounted(50.0, 0.0) == 50.0

def test_average_of_a_few_numbers():
    assert average([10, 20, 30]) == 20.0

def test_average_of_one_element():       # edge case: one element (Lesson 4)
    assert average([42]) == 42.0

def test_average_of_empty_is_zero():     # edge case: empty input (Lesson 4)
    assert average([]) == 0.0

In [ ]:
# Run pytest from the notebook. {sys.executable} is the very Python running this
# kernel, so the command uses the same interpreter you already have.
!"{sys.executable}" -m pytest ../data/lesson5_demo/test_discounts.py -v

### Reading the output

pytest prints one line per test, a `PASSED` (or a dot) for each, and a summary:
`5 passed`. Green means the contract held. Now watch what a *failure* looks like —
we write one deliberately wrong test:

In [ ]:
%%writefile ../data/lesson5_demo/test_discounts_broken.py
"""One deliberately failing test, to see what red looks like."""
from discounts import discounted

def test_intentionally_wrong():
    assert discounted(100.0, 0.10) == 91.0   # the real answer is 90.0

In [ ]:
!"{sys.executable}" -m pytest ../data/lesson5_demo/test_discounts_broken.py -v

🧭 Read the failure bottom-up, exactly like a Unit 4 traceback: pytest shows
`assert 90.0 == 91.0` and both sides of the comparison, so the gap is obvious. A
failing test is not a disaster — it is the tool doing its job, pointing at the exact
line to fix.

### The reveal — you've been graded by this all along

Every auto-graded exercise in this course is what you just wrote: `test_...`
functions full of `assert`, plus edge cases from the Lesson 4 mindset — empty input,
one element, ties. Here is the contract from your *own* Lesson 5 exercise checker:

In [ ]:
checker = Path("../lesson5_exercise/test_lesson5.py").read_text()
print(checker.split('"""')[1])     # the module docstring: the full contract, no spoilers

Notice the edge cases it pins — an empty table, a tie broken by the smaller id, one
row, more customers requested than exist. Those are the Lesson 4 checklist (empty,
one element, ties) turned into tests. The finance assignment's grader is the same
machinery, just dressed to print ✅/❌ and keep running:

In [ ]:
grader = Path("../pandas-finance-assignment/utils/test.py").read_text()
print(grader.split('"""')[1])      # its design-rules docstring

🧭 Same DNA: asserts on known inputs, messages that point at the problem and never
at the solution. You can now *read* any checker in this course and know exactly what
it is asking for.

### Your turn — the refactor-and-test exercise

Open `../lesson5_exercise/`. Inside: `messy_analysis.py` (the ugly-but-correct script
from Unit 3) and `test_lesson5.py` (the contract you just read). Your job: create
`analysis.py` that makes **all 12 tests pass without changing a single number the
script prints** — extracting one function at a time, running pytest after each, and
committing each green step with git (Lesson 3's `add → commit` rhythm). `README.md`
in that folder has the exact commands. One warning it repeats: the edge-case tests
describe inputs the script never actually meets, so answer them by *reading the
script's logic*, not by running it.

> **Unit 5 takeaway:** a test is arrange–act–assert; `assert` is the atom; `pytest`
> runs every `test_...` function from the command line; and the edge cases worth
> writing — empty, one element, ties — come straight from the Lesson 4 mindset. This
> is the machinery behind every checker in the course.

## Wrap-up

### What you can do now

- Write a function as a **contract**: signature, docstring, return value, type hints.
- Tell **pure** from **side-effecting** code — and keep cleaning functions pure.
- Reach for the functional toolkit — comprehensions, `sorted(key=...)`,
  `zip`/`enumerate` — and read a pandas chain as a pipeline.
- Name the smells in your own code and **extract functions** into a module you import.
- Fail well: read tracebacks bottom-up, catch the narrow exception, **raise early**.
- Write tests and run **pytest** — and read any checker in this course.

### Where the earlier lessons went

| Earlier idea | Its Lesson 5 form |
| --- | --- |
| Lesson 1: a type is a claim about legitimate operations | type hints in signatures |
| Lesson 2: interview the data before trusting it | validate early, raise `ValueError` |
| Lesson 2: each verb returns a new frame | method chains as functional pipelines |
| Lesson 3: the `git add → commit` rhythm | commit each green test step |
| Lesson 3: `if __name__ == "__main__"` | scripts that import cleanly so tests can too |
| Lesson 4: algorithm vs implementation | `.apply` (a Python loop) vs vectorization |
| Lesson 4: edge cases — empty, one, ties | the tests worth writing |

### The lesson, one sentence

> *Code you can trust twice is code shaped into named contracts and kept honest by
> tests* — the same move as every earlier lesson: make the properties explicit.

### What comes next

The exercise in `../lesson5_exercise/` turns this reading into muscle. After it, the
course moves into SQL and databases — where "declare the properties, let the engine
run the algorithm," the idea behind Lessons 4 and 5, becomes the entire point.

### Extensions

- [pytest — Get Started](https://docs.pytest.org/en/stable/getting-started.html) — install, write your first test, run it from the terminal
- [Real Python — comprehensions](https://realpython.com/list-comprehension-python/) — the functional toolkit in depth
- [Fluent Python](https://www.fluentpython.com/) (Luciano Ramalho) — functions, the data model, idiomatic Python; selected chapters
- [PEP 8](https://peps.python.org/pep-0008/) — the style guide every Python project assumes